# DLGenAI Project 26T3 - Milestone 4
## Optimization Dynamics, Multi-Level Class Imbalance & Metric Governance

### Instructions & Setup
- **W&B Project**: `24F2004781-t32026`
- **W&B Entity**: `dineshwar-developer-indian-institute-of-technology`
- **Subset**: 1,200 unique ASCII-sorted image IDs (4,800 rows)
- **Split**: 80/20 Stratified on defect combination string (rare < 3 -> `rare_other`) -> 960 train / 240 val images
- **Canvas**: Native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)
- **Fixed Training Budget**: native 128x800 input, AdamW (lr 3e-4), batch size 16, AMP on, seed 42, decision threshold 0.5, no minimum-area filtering.

In [ ]:
# Common Environment & Path Setup
import os
import sys
import time
import gc
import ctypes
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Apply glibc memory optimizations if running on Linux / Kaggle
try:
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-100), ctypes.c_int(0))
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-101), ctypes.c_int(1))
except Exception:
    pass

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

# W&B Configuration
try:
    import wandb
except ImportError:
    wandb = None

class CFG:
    SEED = 42
    WANDB_PROJECT = '24F2004781-t32026'
    WANDB_ENTITY = 'dineshwar-developer-indian-institute-of-technology'
    CANVAS_HEIGHT = 128
    CANVAS_WIDTH = 800
    CANVAS_SIZE = 128 * 800  # 102,400
    DEFECT_CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
    LR = 3e-4
    BATCH_SIZE = 16
    THRESHOLD = 0.5

# Automatic Local vs. Kaggle Path Resolution
def resolve_paths():
    candidates = [
        ('/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/competitions/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/train.csv', '/kaggle/input/industrial-surface-defect-detection/train'),
        ('data/train.csv', 'data/train'),
        ('../data/train.csv', '../data/train'),
    ]
    for csv_p, img_p in candidates:
        if os.path.exists(csv_p):
            return csv_p, img_p
    return 'data/train.csv', 'data/train'

DATA_CSV, IMAGE_DIR = resolve_paths()
print(f'Resolved DATA_CSV: {DATA_CSV}')
print(f'Resolved IMAGE_DIR: {IMAGE_DIR}')


In [ ]:
# Common Setup: Working Subset, Native RLE Decoding & Stratified Split
def rle_to_mask(rle_string, height=128, width=800):
    """Decodes RLE mask on native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)."""
    if pd.isna(rle_string) or not str(rle_string).strip():
        return np.zeros((height, width), dtype=np.uint8)
    canvas_size = height * width
    mask_flat = np.zeros(canvas_size, dtype=np.uint8)
    runs = np.fromstring(str(rle_string), dtype=np.int32, sep=' ')
    starts = runs[0::2] - 1  # Convert 1-based to 0-based
    lengths = runs[1::2]
    for s, l in zip(starts, lengths):
        e = min(s + l, canvas_size)  # Clip runs exceeding canvas
        if s < canvas_size:
            mask_flat[s:e] = 1
    return mask_flat.reshape((height, width), order='F')

# 1. Load train.csv
df_all = pd.read_csv(DATA_CSV)

# 2. Sort unique image IDs alphabetically (ASCII order) and take first 1,200
unique_1200_ids = sorted(df_all['image_id'].unique())[:1200]
df_subset = df_all[df_all['image_id'].isin(unique_1200_ids)].copy().reset_index(drop=True)
print(f'Working subset: {len(unique_1200_ids)} unique image IDs, {len(df_subset)} image-class pairs.')

# 3. Build each image combination string
def get_combo(group):
    defects = group[group['mask_rle'].notna() & (group['mask_rle'].astype(str).str.strip() != '')]['defect_type'].tolist()
    if not defects:
        return 'clean'
    return '+'.join(sorted(defects))

combos = df_subset.groupby('image_id').apply(get_combo)
combo_counts = combos.value_counts()

# Merge combinations with fewer than 3 images into rare_other
rare_combos = combo_counts[combo_counts < 3].index
strat_labels = combos.apply(lambda x: 'rare_other' if x in rare_combos else x)

# 4. 80/20 Stratified Split with random_state=42
train_image_ids, val_image_ids = train_test_split(
    list(strat_labels.index),
    test_size=0.20,
    random_state=42,
    stratify=strat_labels.tolist()
)
train_image_ids = set(train_image_ids)
val_image_ids = set(val_image_ids)
print(f'Split complete: {len(train_image_ids)} train images / {len(val_image_ids)} validation images.')


---
## Question 1: Image-Level and Class-Level Imbalance

Use the 1,200-image subset (the full subset, not the split).

### Question:
(a) How many images are completely clean?
(b) How many images are positive for each of `defect_a`, `defect_b`, `defect_c`, `defect_d`?
(c) How many unique combination strings exist (before any merging)?
(d) What are the sizes of the largest and the smallest combination group?

**Answer format:** `a; b_a, b_b, b_c, b_d; c; d_largest, d_smallest` (all integers).


In [ ]:
# ==============================================================================
# Question 1: Image-Level and Class-Level Imbalance
# ==============================================================================
# Check positive status for each row
df_subset['is_positive'] = df_subset['mask_rle'].notna() & (df_subset['mask_rle'].astype(str).str.strip() != '')

# (a) Completely clean images
pos_per_image = df_subset.groupby('image_id')['is_positive'].sum()
ans_a = int((pos_per_image == 0).sum())

# (b) Positive images for each defect type
b_counts = {}
for defect in CFG.DEFECT_CLASSES:
    cnt = df_subset[(df_subset['defect_type'] == defect) & df_subset['is_positive']]['image_id'].nunique()
    b_counts[defect] = int(cnt)
ans_b_a = b_counts['defect_a']
ans_b_b = b_counts['defect_b']
ans_b_c = b_counts['defect_c']
ans_b_d = b_counts['defect_d']

# (c) Unique combination strings (before merging)
ans_c = int(combos.nunique())

# (d) Largest and smallest combination group sizes
ans_d_largest = int(combo_counts.max())
ans_d_smallest = int(combo_counts.min())

print(f'(a) Completely clean images: {ans_a}')
print(f'(b) Positive counts per defect: a={ans_b_a}, b={ans_b_b}, c={ans_b_c}, d={ans_b_d}')
print(f'(c) Unique combinations: {ans_c}')
print(f'(d) Largest group size: {ans_d_largest}, Smallest group size: {ans_d_smallest}')

# Final Formatted Answer
q1_answer = f'{ans_a}; {ans_b_a}, {ans_b_b}, {ans_b_c}, {ans_b_d}; {ans_c}; {ans_d_largest}, {ans_d_smallest}'
print('\n' + '='*40)
print(f'Question 1 Final Answer: {q1_answer}')
print('='*40)
